In [2]:
# 1. Import required libraries
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Upload the file from your computer to Colab
print("Please upload your telco_churn.csv file:")
uploaded = files.upload()

# 3. Load the dataset into a Pandas DataFrame
# (Make sure the filename perfectly matches what you named it)
df = pd.read_csv('telco_churn.csv')

# 4. View the first 5 rows
display(df.head())

Please upload your telco_churn.csv file:


Saving telco_churn.csv to telco_churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. FIX HIDDEN MISSING VALUES
# Convert 'TotalCharges' to numeric and drop the 11 missing rows
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# Drop 'customerID' as it has no predictive power
df.drop('customerID', axis=1, inplace=True)

# 2. FEATURE ENGINEERING (Task 2 Requirement)
# Let's create a new feature: 'Total_Services'
# This counts how many extra services a customer has subscribed to
services = ['PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies']

# We will count how many times 'Yes' appears for these services per customer
df['Total_Services'] = df[services].apply(lambda x: (x == 'Yes').sum(), axis=1)

# 3. ENCODING CATEGORICAL VARIABLES
# Convert binary 'Yes'/'No' columns to 1 and 0
binary_columns = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Convert female/male to 1/0
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})

# Use One-Hot Encoding for the remaining categorical columns with multiple options (like 'Contract')
df = pd.get_dummies(df, drop_first=True)

# 4. SCALING NUMERICAL FEATURES
# Machine learning models perform best when continuous numbers are scaled between 0 and 1
scaler = MinMaxScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Total_Services']

df[num_cols] = scaler.fit_transform(df[num_cols])

# 5. CHECK THE FINAL PROCESSED DATA
print("Preprocessing Complete! Final Dataset Shape:", df.shape)
display(df.head())

Preprocessing Complete! Final Dataset Shape: (7032, 32)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,0.000000,0,1,0.115423,0.001275,0,...,False,False,False,False,False,False,False,False,True,False
1,0,0,0,0,0.464789,1,0,0.385075,0.215867,0,...,False,False,False,False,False,True,False,False,False,True
2,0,0,0,0,0.014085,1,1,0.354229,0.010310,1,...,False,False,False,False,False,False,False,False,False,True
3,0,0,0,0,0.619718,0,0,0.239303,0.210241,0,...,True,False,False,False,False,True,False,False,False,False
4,1,0,0,0,0.014085,1,1,0.521891,0.015330,1,...,False,False,False,False,False,False,False,False,True,False


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. DEFINE FEATURES (X) AND TARGET (y)
# 'Churn' is what we want to predict. X is everything else.
X = df.drop('Churn', axis=1)
y = df['Churn']

# 2. SPLIT THE DATA
# We use 80% of the data to train the model, and 20% to test its accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}\n")

# 3. TRAIN MODEL 1: LOGISTIC REGRESSION (Baseline)
print("Training Logistic Regression...")
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_predictions = log_model.predict(X_test)

# 4. TRAIN MODEL 2: RANDOM FOREST (Advanced)
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# 5. COMPARE METRICS (Task 4 Requirement)
print("\n" + "="*40)
print("--- LOGISTIC REGRESSION RESULTS ---")
print("Accuracy:", accuracy_score(y_test, log_predictions))
print(classification_report(y_test, log_predictions))

print("="*40)
print("--- RANDOM FOREST RESULTS ---")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print(classification_report(y_test, rf_predictions))

Training data shape: (5625, 31)
Testing data shape: (1407, 31)

Training Logistic Regression...
Training Random Forest...

--- LOGISTIC REGRESSION RESULTS ---
Accuracy: 0.7874911158493249
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

--- RANDOM FOREST RESULTS ---
Accuracy: 0.7839374555792467
              precision    recall  f1-score   support

           0       0.82      0.90      0.86      1033
           1       0.62      0.47      0.54       374

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.70      1407
weighted avg       0.77      0.78      0.77      1407



In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("Building Artificial Neural Network (ANN)...")

# 1. INITIALIZE THE NEURAL NETWORK
ann_model = Sequential()

# 2. ADD LAYERS
# Input layer and first hidden layer (using ReLU activation)
# The 'input_shape' must match the number of features in your dataset (31)
ann_model.add(Dense(units=32, activation='relu', input_shape=(X_train.shape[1],)))
ann_model.add(Dropout(0.2)) # Dropout prevents overfitting

# Second hidden layer
ann_model.add(Dense(units=16, activation='relu'))
ann_model.add(Dropout(0.2))

# Output layer (using Sigmoid because it's a binary yes/no prediction)
ann_model.add(Dense(units=1, activation='sigmoid'))

# 3. COMPILE THE MODEL
ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. TRAIN THE MODEL
# We use epochs=50 (it will pass through the data 50 times)
print("Training ANN...")
history = ann_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# 5. EVALUATE THE MODEL
print("\n" + "="*40)
print("--- DEEP LEARNING (ANN) RESULTS ---")
ann_loss, ann_accuracy = ann_model.evaluate(X_test, y_test)
print(f"ANN Accuracy: {ann_accuracy}")

# Get detailed classification report
ann_predictions = (ann_model.predict(X_test) > 0.5).astype("int32")
print(classification_report(y_test, ann_predictions))

Building Artificial Neural Network (ANN)...
Training ANN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7262 - loss: 0.5257 - val_accuracy: 0.7719 - val_loss: 0.4590
Epoch 2/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7778 - loss: 0.4597 - val_accuracy: 0.7818 - val_loss: 0.4471
Epoch 3/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7858 - loss: 0.4430 - val_accuracy: 0.7811 - val_loss: 0.4403
Epoch 4/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7948 - loss: 0.4345 - val_accuracy: 0.7875 - val_loss: 0.4398
Epoch 5/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7947 - loss: 0.4344 - val_accuracy: 0.7832 - val_loss: 0.4378
Epoch 6/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7959 - loss: 0.4313 - val_accuracy: 0.7832 - val_loss: 0.4378
Epoch 7/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7988 - loss: 0.4298 - val_accuracy: 0.7804 - val_loss: 0.4388
Epoch 8/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7982 - loss: 0.4249 - val_accuracy: 0.

In [6]:
import joblib

# 1. Save the winning model
joblib.dump(log_model, 'churn_model.pkl')

# 2. Save the scaler (Crucial! New data must be scaled the exact same way)
joblib.dump(scaler, 'scaler.pkl')

print("Model and Scaler successfully saved!")

Model and Scaler successfully saved!
